In [1]:
from pathlib import Path
import pandas as pd
import gzip

data_path = Path("../data/raw/GSE75037_series_matrix.txt.gz")

print("File exists:", data_path.exists())
print("File size (MB):", round(data_path.stat().st_size / (1024**2), 2))

File exists: True
File size (MB): 14.88


In [2]:
with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for i in range(15):
        print(f.readline().rstrip())

!Series_title	"Expression profiling of 83 matched pairs of lung adenocarcinomas and non-malignant adjacent tissue"
!Series_geo_accession	"GSE75037"
!Series_status	"Public on Jun 01 2016"
!Series_submission_date	"Nov 16 2015"
!Series_last_update_date	"Feb 18 2019"
!Series_pubmed_id	"27354471"
!Series_summary	"Background: Non-small cell lung cancers (NSCLCs) consist of adenocarcinoma (ADC), squamous cell carcinoma (SCC) and other types. Since most NSCLCs are now diagnosed from small biopsies or cytology materials, classification is not always accurate. This is a problem as many therapy regimens and clinical trials are histology-dependent."
!Series_summary	""
!Series_summary	"Specific Aim: To develop an RNA expression signature as an adjunct test for routine histo-pathological classification of NSCLCs."
!Series_summary	""
!Series_summary	"Methods: A microarray dataset of resected ADC and SCC cases was used as the learning set for an ADC-SCC signature. The Cancer Genome Atlas (TCGA) lung R

In [3]:
with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"):
            print(line.rstrip())
        elif line.startswith("!Sample_title"):
            print(line.rstrip())
        elif line.startswith("!Sample_characteristics_ch1"):
            print(line.rstrip())

!Sample_title	"05L4_N"	"05L4_T"	"05L6_N"	"05L6_T"	"05L7_N"	"05L7_T"	"05L9_N"	"05L9_T"	"05L10_N"	"05L10_T"	"05L12_N"	"05L12_T"	"05L13_N"	"05L13_T"	"05L19_N"	"05L19_T"	"05L27_N"	"05L27_T"	"05L32_N"	"05L32_T"	"05L33_N"	"05L33_T"	"05L34_N"	"05L34_T"	"05L38_N"	"05L38_T"	"05L39_N"	"05L39_T"	"05L40_N"	"05L40_T"	"05L43_N"	"05L43_T"	"05L45_N"	"05L45_T"	"05L46_N"	"05L46_T"	"05L47_N"	"05L47_T"	"05L49_N"	"05L49_T"	"05L52_N"	"05L52_T"	"05L54_N"	"05L54_T"	"06L3_N"	"06L3_T"	"06L10_N"	"06L10_T"	"06L11_N"	"06L11_T"	"06L14_N"	"06L14_T"	"06L20_N"	"06L20_T"	"06L22_N"	"06L22_T"	"06L29_N"	"06L29_T"	"06L30_N"	"06L30_T"	"06L39_N"	"06L39_T"	"06L42_N"	"06L42_T"	"06L43_N"	"06L43_T"	"06L45_N"	"06L45_T"	"06L46_N"	"06L46_T"	"06L50_N"	"06L50_T"	"06L52_N"	"06L52_T"	"06L53_N"	"06L53_T"	"06L54_N"	"06L54_T"	"06L57_N"	"06L57_T"	"06L64_N"	"06L64_T"	"06L71_N"	"06L71_T"	"06L74_N"	"06L74_T"	"06L75_N"	"06L75_T"	"07L5_N"	"07L5_T"	"07L6_N"	"07L6_T"	"07L13_N"	"07L13_T"	"07L14_N"	"07L14_T"	"07L16_N"	"07L16_T"	"07L18_N"	"07L18_T"	

In [4]:
import re

sample_accessions = []
sample_characteristics = []

with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"):
            sample_accessions = line.rstrip().split("\t")[1:]
        
        elif line.startswith("!Sample_characteristics_ch1"):
            sample_characteristics.append(
                line.rstrip().split("\t")[1:]
            )

print("Number of samples:", len(sample_accessions))
print("Number of characteristic rows:", len(sample_characteristics))

Number of samples: 166
Number of characteristic rows: 10


In [5]:
# Find the metadata row containing histology information
histology_row = None

for row in sample_characteristics:
    if any("histology:" in value.lower() for value in row):
        histology_row = row
        break

print("Histology metadata found:", histology_row is not None)

# Extract the histology label from each sample
histology = []

for value in histology_row:
    match = re.search(r"histology:\s*(.*)", value, re.IGNORECASE)
    histology.append(match.group(1) if match else "Unknown")

# Create a clean metadata table
metadata = pd.DataFrame({
    "sample_id": sample_accessions,
    "histology": histology
})

print(metadata.head())
print()
print("Histology counts:")
print(metadata["histology"].value_counts())

Histology metadata found: True
      sample_id        histology
0  "GSM1941120"   Non-malignant"
1  "GSM1941121"  Adenocarcinoma"
2  "GSM1941122"   Non-malignant"
3  "GSM1941123"  Adenocarcinoma"
4  "GSM1941124"   Non-malignant"

Histology counts:
histology
Non-malignant"     83
Adenocarcinoma"    83
Name: count, dtype: int64


In [6]:
# Inspect GEO sample metadata fields related to sample identity and pairing

metadata_rows = {}

with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for line in f:
        if line.startswith("!Sample_"):
            parts = line.rstrip().split("\t")
            field = parts[0]
            values = parts[1:]
            metadata_rows[field] = values

print("Relevant metadata fields:\n")

for field in metadata_rows:
    if any(keyword in field.lower()
           for keyword in ["title", "source", "patient", "subject", "characteristics"]):
        print(field)

Relevant metadata fields:

!Sample_title
!Sample_source_name_ch1
!Sample_characteristics_ch1


In [7]:
print("SAMPLE TITLES:\n")

for value in metadata_rows["!Sample_title"][:20]:
    print(value)

print("\nSAMPLE SOURCES:\n")

for value in metadata_rows["!Sample_source_name_ch1"][:20]:
    print(value)

SAMPLE TITLES:

"05L4_N"
"05L4_T"
"05L6_N"
"05L6_T"
"05L7_N"
"05L7_T"
"05L9_N"
"05L9_T"
"05L10_N"
"05L10_T"
"05L12_N"
"05L12_T"
"05L13_N"
"05L13_T"
"05L19_N"
"05L19_T"
"05L27_N"
"05L27_T"
"05L32_N"
"05L32_T"

SAMPLE SOURCES:

"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"
"Non-malignant lung"
"Lung cancer"


In [8]:
# Build a clean metadata table from GEO sample information

sample_titles = [
    value.strip('"')
    for value in metadata_rows["!Sample_title"]
]

metadata = pd.DataFrame({
    "sample_id": sample_accessions,
    "sample_title": sample_titles,
    "histology": histology
})

# Extract the patient/pair identifier from the sample title
metadata["pair_id"] = metadata["sample_title"].str.replace(
    r"_[NT]$", "", regex=True
)

# Extract tumor/normal status from the sample title
metadata["status_from_title"] = metadata["sample_title"].str.extract(
    r"_([NT])$"
)[0].map({
    "T": "Tumor",
    "N": "Normal"
})

# Display the first 20 samples
metadata.head(20)

,sample_id,sample_title,histology,pair_id,status_from_title
0,"""GSM1941120""",05L4_N,"Non-malignant""",05L4,Normal
1,"""GSM1941121""",05L4_T,"Adenocarcinoma""",05L4,Tumor
2,"""GSM1941122""",05L6_N,"Non-malignant""",05L6,Normal
3,"""GSM1941123""",05L6_T,"Adenocarcinoma""",05L6,Tumor
4,"""GSM1941124""",05L7_N,"Non-malignant""",05L7,Normal
5,"""GSM1941125""",05L7_T,"Adenocarcinoma""",05L7,Tumor
6,"""GSM1941126""",05L9_N,"Non-malignant""",05L9,Normal
7,"""GSM1941127""",05L9_T,"Adenocarcinoma""",05L9,Tumor
8,"""GSM1941128""",05L10_N,"Non-malignant""",05L10,Normal
9,"""GSM1941129""",05L10_T,"Adenocarcinoma""",05L10,Tumor


In [9]:
# Verify the matched tumor-normal structure

pair_check = (
    metadata.groupby("pair_id")["status_from_title"]
    .value_counts()
    .unstack(fill_value=0)
)

print("Number of unique pairs:", pair_check.shape[0])
print()

print("Pairs with exactly 1 tumor and 1 normal:")
valid_pairs = (
    (pair_check["Tumor"] == 1) &
    (pair_check["Normal"] == 1)
)

print(valid_pairs.sum())

print()
print("Any incomplete pairs:", (~valid_pairs).sum())

Number of unique pairs: 83

Pairs with exactly 1 tumor and 1 normal:
83

Any incomplete pairs: 0


In [10]:
# Locate the beginning and end of the expression matrix

table_begin = None
table_end = None

with gzip.open(data_path, "rt", encoding="utf-8") as f:
    for line_number, line in enumerate(f):
        if line.startswith("!series_matrix_table_begin"):
            table_begin = line_number
        elif line.startswith("!series_matrix_table_end"):
            table_end = line_number
            break

print("Expression table begins at line:", table_begin)
print("Expression table ends at line:", table_end)
print("Number of lines in expression table:", table_end - table_begin - 1)

Expression table begins at line: 80
Expression table ends at line: 48885
Number of lines in expression table: 48804


In [11]:
# Load the processed GEO expression matrix

expression = pd.read_csv(
    data_path,
    sep="\t",
    comment="!",
    index_col=0
)

print("Expression matrix shape:", expression.shape)
print()
print("First 5 rows:")
display(expression.head())

Expression matrix shape: (48803, 166)

First 5 rows:


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124,GSM1941125,GSM1941126,GSM1941127,GSM1941128,GSM1941129,...,GSM1941276,GSM1941277,GSM1941278,GSM1941279,GSM1941280,GSM1941281,GSM1941282,GSM1941283,GSM1941284,GSM1941285
ID_REF,,,,,,,,,,,,,,,,,,,,,
ILMN_1343291,15.343,15.403,15.446,15.140,15.254,14.984,15.318,15.105,15.403,15.204,...,15.386,14.948,15.343,15.264,15.318,15.307,15.422,15.494,14.998,15.123
ILMN_1343295,12.280,13.753,12.342,11.625,12.518,12.007,12.274,13.339,12.178,13.020,...,11.914,13.520,12.206,13.103,12.244,13.536,11.929,14.053,12.158,13.738
ILMN_1651199,3.350,3.561,3.459,4.154,3.848,3.700,3.336,4.278,3.766,3.755,...,3.954,3.350,3.263,3.154,3.897,3.632,3.573,3.644,3.632,3.121
ILMN_1651209,3.711,4.684,5.104,4.771,4.087,5.406,4.684,4.717,3.945,5.389,...,4.162,5.739,4.597,5.722,4.365,5.567,3.945,5.624,4.907,5.612
ILMN_1651210,3.573,3.621,3.807,4.406,3.459,3.350,3.722,3.711,4.044,3.755,...,4.138,3.406,3.406,3.667,3.818,3.766,3.336,3.365,3.446,3.667


In [12]:
# Check whether expression-matrix samples match our metadata

expression_samples = expression.columns.astype(str).str.strip('"')
metadata_samples = pd.Series(metadata["sample_id"]).astype(str).str.strip('"')

print("Expression samples:", len(expression_samples))
print("Metadata samples:", len(metadata_samples))

print("\nSame sample IDs:", set(expression_samples) == set(metadata_samples))
print("Same order:", list(expression_samples) == list(metadata_samples))

Expression samples: 166
Metadata samples: 166

Same sample IDs: True
Same order: True


In [13]:
print("Expression matrix shape:", expression.shape)

print("\nData types:")
print(expression.dtypes.value_counts())

print("\nMissing values:", expression.isna().sum().sum())

print("\nExpression value range:")
print("Minimum:", expression.min().min())
print("Maximum:", expression.max().max())

print("\nSummary statistics:")
display(expression.iloc[:, :5].describe())

Expression matrix shape: (48803, 166)

Data types:
float64    166
Name: count, dtype: int64

Missing values: 0

Expression value range:
Minimum: 2.787
Maximum: 15.494

Summary statistics:


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124
count,48803.000000,48803.000000,48803.000000,48803.000000,48803.000000
mean,5.354239,5.354242,5.354427,5.354252,5.353810
std,2.475766,2.475689,2.475572,2.475748,2.476106
min,2.807000,2.787000,2.787000,2.787000,2.787000
25%,3.632000,3.644000,3.632000,3.621000,3.632000
50%,4.121000,4.129000,4.138000,4.121000,4.121000
75%,6.600000,6.597000,6.595000,6.592000,6.594000
max,15.494000,15.494000,15.494000,15.494000,15.494000


In [14]:
print("Shape:", expression.shape)

print("\nData types:")
print(expression.dtypes.value_counts())

print("\nMissing values:", expression.isna().sum().sum())

print("\nMinimum expression:", expression.min().min())
print("Maximum expression:", expression.max().max())

print("\nFirst 5 probes:")
display(expression.head())

Shape: (48803, 166)

Data types:
float64    166
Name: count, dtype: int64

Missing values: 0

Minimum expression: 2.787
Maximum expression: 15.494

First 5 probes:


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124,GSM1941125,GSM1941126,GSM1941127,GSM1941128,GSM1941129,...,GSM1941276,GSM1941277,GSM1941278,GSM1941279,GSM1941280,GSM1941281,GSM1941282,GSM1941283,GSM1941284,GSM1941285
ID_REF,,,,,,,,,,,,,,,,,,,,,
ILMN_1343291,15.343,15.403,15.446,15.140,15.254,14.984,15.318,15.105,15.403,15.204,...,15.386,14.948,15.343,15.264,15.318,15.307,15.422,15.494,14.998,15.123
ILMN_1343295,12.280,13.753,12.342,11.625,12.518,12.007,12.274,13.339,12.178,13.020,...,11.914,13.520,12.206,13.103,12.244,13.536,11.929,14.053,12.158,13.738
ILMN_1651199,3.350,3.561,3.459,4.154,3.848,3.700,3.336,4.278,3.766,3.755,...,3.954,3.350,3.263,3.154,3.897,3.632,3.573,3.644,3.632,3.121
ILMN_1651209,3.711,4.684,5.104,4.771,4.087,5.406,4.684,4.717,3.945,5.389,...,4.162,5.739,4.597,5.722,4.365,5.567,3.945,5.624,4.907,5.612
ILMN_1651210,3.573,3.621,3.807,4.406,3.459,3.350,3.722,3.711,4.044,3.755,...,4.138,3.406,3.406,3.667,3.818,3.766,3.336,3.365,3.446,3.667


In [15]:
expression_samples = expression.columns.astype(str).str.strip('"')
metadata_samples = metadata["sample_id"].astype(str).str.strip('"')

print("Expression samples:", len(expression_samples))
print("Metadata samples:", len(metadata_samples))

print("\nSame sample IDs:",
      set(expression_samples) == set(metadata_samples))

print("Same order:",
      list(expression_samples) == list(metadata_samples))

Expression samples: 166
Metadata samples: 166

Same sample IDs: True
Same order: True


In [16]:
print("Shape:", expression.shape)

print("\nData types:")
print(expression.dtypes.value_counts())

print("\nMissing values:", expression.isna().sum().sum())

print("\nMinimum expression:", expression.min().min())
print("Maximum expression:", expression.max().max())

print("\nFirst 5 probes:")
display(expression.head())

Shape: (48803, 166)

Data types:
float64    166
Name: count, dtype: int64

Missing values: 0

Minimum expression: 2.787
Maximum expression: 15.494

First 5 probes:


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124,GSM1941125,GSM1941126,GSM1941127,GSM1941128,GSM1941129,...,GSM1941276,GSM1941277,GSM1941278,GSM1941279,GSM1941280,GSM1941281,GSM1941282,GSM1941283,GSM1941284,GSM1941285
ID_REF,,,,,,,,,,,,,,,,,,,,,
ILMN_1343291,15.343,15.403,15.446,15.140,15.254,14.984,15.318,15.105,15.403,15.204,...,15.386,14.948,15.343,15.264,15.318,15.307,15.422,15.494,14.998,15.123
ILMN_1343295,12.280,13.753,12.342,11.625,12.518,12.007,12.274,13.339,12.178,13.020,...,11.914,13.520,12.206,13.103,12.244,13.536,11.929,14.053,12.158,13.738
ILMN_1651199,3.350,3.561,3.459,4.154,3.848,3.700,3.336,4.278,3.766,3.755,...,3.954,3.350,3.263,3.154,3.897,3.632,3.573,3.644,3.632,3.121
ILMN_1651209,3.711,4.684,5.104,4.771,4.087,5.406,4.684,4.717,3.945,5.389,...,4.162,5.739,4.597,5.722,4.365,5.567,3.945,5.624,4.907,5.612
ILMN_1651210,3.573,3.621,3.807,4.406,3.459,3.350,3.722,3.711,4.044,3.755,...,4.138,3.406,3.406,3.667,3.818,3.766,3.336,3.365,3.446,3.667


In [17]:
annotation_path = Path(
    "../data/raw/GPL6884_HumanWG-6_V3_0_R0_11282955_A.bgx.gz"
)

print("File exists:", annotation_path.exists())
print("File size (MB):", round(annotation_path.stat().st_size / (1024**2), 2))

File exists: True
File size (MB): 6.27


In [18]:
with gzip.open(annotation_path, "rt", encoding="utf-8") as f:
    for i in range(15):
        print(f.readline().rstrip())

? Illumina, Inc.
[Heading]
Date	5/1/2008
ContentVersion	1.0
FormatVersion	1.0.0
Number of Probes	48804
Number of Controls	785
[Probes]
Species	Source	Search_Key	Transcript	ILMN_Gene	Source_Reference_ID	RefSeq_ID	Unigene_ID	Entrez_Gene_ID	GI	Accession	Symbol	Protein_Product	Probe_Id	Array_Address_Id	Probe_Type	Probe_Start	Probe_Sequence	Chromosome	Probe_Chr_Orientation	Probe_Coordinates	Cytoband	Definition	Ontology_Component	Ontology_Process	Ontology_Function	Synonyms	Obsolete_Probe_Id
Homo sapiens	Unigene	ILMN_89282	ILMN_89282	HS.388528	Hs.388528		Hs.388528		23525203	BU678343			ILMN_1825594	0001740241	S	349	CTCTCTAAAGGGACAACAGAGTGGACAGTCAAGGAACTCCACATATTCAT					UI-CF-EC0-abi-c-12-0-UI.s1 UI-CF-EC0 Homo sapiens cDNA clone UI-CF-EC0-abi-c-12-0-UI 3, mRNA sequence
Homo sapiens	RefSeq	ILMN_35826	ILMN_35826	LOC441782	XM_497527.2	XM_497527.2		441782	89042416	XM_497527.2	LOC441782	XP_497527.2	ILMN_1810803	0001850750	S	902	GGGGTCAAGCCCAGGTGAAATGTGGATTGGAAAAGTGCTTCCCTTGCCCC					PREDICTED: Homo 

In [19]:
import io

with gzip.open(annotation_path, "rt", encoding="utf-8") as f:
    lines = f.readlines()

# Find the Probes section
probes_start = lines.index("[Probes]\n") + 1

# Find where the next section begins
probes_end = next(
    i for i in range(probes_start, len(lines))
    if lines[i].startswith("[") and i != probes_start
)

probe_text = "".join(lines[probes_start:probes_end])

annotation = pd.read_csv(
    io.StringIO(probe_text),
    sep="\t"
)

print("Annotation shape:", annotation.shape)
print("\nAnnotation columns:")
print(annotation.columns.tolist())

Annotation shape: (48804, 28)

Annotation columns:
['Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID', 'Unigene_ID', 'Entrez_Gene_ID', 'GI', 'Accession', 'Symbol', 'Protein_Product', 'Probe_Id', 'Array_Address_Id', 'Probe_Type', 'Probe_Start', 'Probe_Sequence', 'Chromosome', 'Probe_Chr_Orientation', 'Probe_Coordinates', 'Cytoband', 'Definition', 'Ontology_Component', 'Ontology_Process', 'Ontology_Function', 'Synonyms', 'Obsolete_Probe_Id']


In [20]:
print("\nFirst 5 annotation rows:")
display(annotation.head())

print("\nSearch_Key examples:")
print(annotation["Search_Key"].head().tolist())


First 5 annotation rows:


,Species,Source,Search_Key,Transcript,ILMN_Gene,Source_Reference_ID,RefSeq_ID,Unigene_ID,Entrez_Gene_ID,GI,...,Chromosome,Probe_Chr_Orientation,Probe_Coordinates,Cytoband,Definition,Ontology_Component,Ontology_Process,Ontology_Function,Synonyms,Obsolete_Probe_Id
0,Homo sapiens,Unigene,ILMN_89282,ILMN_89282,HS.388528,Hs.388528,NaN,Hs.388528,NaN,23525203,...,NaN,NaN,NaN,NaN,UI-CF-EC0-abi-c-12-0-UI.s1 UI-CF-EC0 Homo sapi...,NaN,NaN,NaN,NaN,NaN
1,Homo sapiens,RefSeq,ILMN_35826,ILMN_35826,LOC441782,XM_497527.2,XM_497527.2,NaN,441782.0,89042416,...,NaN,NaN,NaN,NaN,PREDICTED: Homo sapiens similar to spectrin do...,NaN,NaN,NaN,NaN,NaN
2,Homo sapiens,RefSeq,ILMN_25544,ILMN_25544,JMJD1A,NM_018433.3,NM_018433.3,NaN,55818.0,46358420,...,2,+,86572991-86573040,2p11.2e,Homo sapiens jumonji domain containing 1A (JMJ...,nucleus [goid 5634] [evidence IEA],chromatin modification [goid 16568] [evidence ...,oxidoreductase activity [goid 16491] [evidence...,JHMD2A; JMJD1; TSGA; KIAA0742; DKFZp686A24246;...,NaN
3,Homo sapiens,Unigene,ILMN_132331,ILMN_132331,HS.580150,Hs.580150,NaN,Hs.580150,NaN,7376124,...,NaN,NaN,NaN,NaN,hi56g05.x1 Soares_NFL_T_GBC_S1 Homo sapiens cD...,NaN,NaN,NaN,NaN,NaN
4,Homo sapiens,Unigene,ILMN_105017,ILMN_105017,HS.540210,Hs.540210,NaN,Hs.540210,NaN,5437312,...,NaN,NaN,NaN,NaN,wk77d04.x1 NCI_CGAP_Pan1 Homo sapiens cDNA clo...,NaN,NaN,NaN,NaN,NaN



Search_Key examples:
['ILMN_89282', 'ILMN_35826', 'ILMN_25544', 'ILMN_132331', 'ILMN_105017']


In [21]:
# Keep only the columns we need for probe-to-gene mapping
probe_annotation = annotation[
    ["Search_Key", "Symbol", "Entrez_Gene_ID"]
].copy()

# Clean probe IDs
probe_annotation["Search_Key"] = (
    probe_annotation["Search_Key"]
    .astype(str)
    .str.strip()
)

# Treat missing symbols as missing values
probe_annotation["Symbol"] = (
    probe_annotation["Symbol"]
    .replace(["", "nan", "NaN"], pd.NA)
)

print("Total annotation probes:", len(probe_annotation))
print("Probes with gene symbols:", probe_annotation["Symbol"].notna().sum())
print("Probes without gene symbols:", probe_annotation["Symbol"].isna().sum())

print("\nExample mappings:")
display(probe_annotation.head(10))

Total annotation probes: 48804
Probes with gene symbols: 35967
Probes without gene symbols: 12837

Example mappings:


,Search_Key,Symbol,Entrez_Gene_ID
0,ILMN_89282,NaN,NaN
1,ILMN_35826,LOC441782,441782.0
2,ILMN_25544,JMJD1A,55818.0
3,ILMN_132331,NaN,NaN
4,ILMN_105017,NaN,NaN
5,ILMN_75398,NaN,NaN
6,ILMN_10519,NCOA3,8202.0
7,ILMN_117209,NaN,NaN
8,ILMN_17234,LOC389834,389834.0
9,ILMN_19244,C17orf77,146723.0


In [22]:
# Check how many of our expression probes have annotations
expression_probes = pd.Index(expression.index.astype(str))

matched = expression_probes.isin(
    probe_annotation["Search_Key"]
)

print("Expression probes:", len(expression_probes))
print("Annotated probes:", matched.sum())
print("Unannotated probes:", (~matched).sum())

print("\nPercentage annotated:",
      round(matched.mean() * 100, 2), "%")

Expression probes: 48803
Annotated probes: 0
Unannotated probes: 48803

Percentage annotated: 0.0 %


In [23]:
# Prepare expression matrix for annotation
expression_reset = expression.reset_index()
expression_reset = expression_reset.rename(columns={"ID_REF": "Search_Key"})

# Clean probe IDs
expression_reset["Search_Key"] = (
    expression_reset["Search_Key"]
    .astype(str)
    .str.strip()
)

# Merge expression data with probe annotation
expression_annotated = expression_reset.merge(
    probe_annotation,
    on="Search_Key",
    how="left"
)

print("Expression rows:", len(expression_reset))
print("Rows after annotation:", len(expression_annotated))

print("\nMissing gene symbols:",
      expression_annotated["Symbol"].isna().sum())

print("\nExample:")
display(
    expression_annotated[
        ["Search_Key", "Symbol", "Entrez_Gene_ID"]
    ].head(10)
)

Expression rows: 48803
Rows after annotation: 48803

Missing gene symbols: 48803

Example:


C:\Users\DELL\AppData\Local\Temp\ipykernel_2208\4085354487.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  expression_reset = expression.reset_index()


,Search_Key,Symbol,Entrez_Gene_ID
0,ILMN_1343291,NaN,NaN
1,ILMN_1343295,NaN,NaN
2,ILMN_1651199,NaN,NaN
3,ILMN_1651209,NaN,NaN
4,ILMN_1651210,NaN,NaN
5,ILMN_1651221,NaN,NaN
6,ILMN_1651228,NaN,NaN
7,ILMN_1651229,NaN,NaN
8,ILMN_1651230,NaN,NaN
9,ILMN_1651232,NaN,NaN


In [24]:
# Keep only probes that map to a gene symbol
expression_annotated = expression_annotated[
    expression_annotated["Symbol"].notna()
].copy()

# Remove obviously invalid placeholder symbols
invalid_symbols = ["", "---", "NA", "N/A"]

expression_annotated = expression_annotated[
    ~expression_annotated["Symbol"].isin(invalid_symbols)
].copy()

print("Rows with valid gene symbols:",
      len(expression_annotated))

print("Unique gene symbols:",
      expression_annotated["Symbol"].nunique())

print("Duplicate-probe rows:",
      len(expression_annotated) -
      expression_annotated["Symbol"].nunique())

Rows with valid gene symbols: 0
Unique gene symbols: 0
Duplicate-probe rows: 0


In [25]:
# Expression columns
sample_columns = expression.columns.tolist()

# Calculate variance of each probe across all samples
expression_annotated["probe_variance"] = (
    expression_annotated[sample_columns]
    .var(axis=1)
)

# Highest-variance probe for each gene
best_probes = (
    expression_annotated
    .sort_values("probe_variance", ascending=False)
    .drop_duplicates("Symbol")
    .copy()
)

print("Genes retained:", best_probes["Symbol"].nunique())
print("Rows retained:", len(best_probes))

print("\nExample selected probes:")
display(
    best_probes[
        ["Search_Key", "Symbol", "probe_variance"]
    ].head(10)
)

Genes retained: 0
Rows retained: 0

Example selected probes:


,Search_Key,Symbol,probe_variance


In [26]:
gene_expression = best_probes.set_index("Symbol")[sample_columns].copy()

print("Gene-expression matrix shape:",
      gene_expression.shape)

print("\nFirst 5 genes:")
display(gene_expression.head())

Gene-expression matrix shape: (0, 166)

First 5 genes:


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124,GSM1941125,GSM1941126,GSM1941127,GSM1941128,GSM1941129,...,GSM1941276,GSM1941277,GSM1941278,GSM1941279,GSM1941280,GSM1941281,GSM1941282,GSM1941283,GSM1941284,GSM1941285
Symbol,,,,,,,,,,,,,,,,,,,,,


In [27]:
print("Expression probe examples:")
print(expression_reset["Search_Key"].head(10).tolist())

print("\nAnnotation probe examples:")
print(probe_annotation["Search_Key"].head(10).tolist())

print("\nExpression probe dtype:")
print(expression_reset["Search_Key"].dtype)

print("\nAnnotation probe dtype:")
print(probe_annotation["Search_Key"].dtype)

print("\nDirect matches:")
print(
    expression_reset["Search_Key"]
    .isin(probe_annotation["Search_Key"])
    .sum()
)

print("\nAnnotation symbols:")
print(probe_annotation["Symbol"].head(10).tolist())

print("\nNon-missing symbols:")
print(probe_annotation["Symbol"].notna().sum())


Expression probe examples:
['ILMN_1343291', 'ILMN_1343295', 'ILMN_1651199', 'ILMN_1651209', 'ILMN_1651210', 'ILMN_1651221', 'ILMN_1651228', 'ILMN_1651229', 'ILMN_1651230', 'ILMN_1651232']

Annotation probe examples:
['ILMN_89282', 'ILMN_35826', 'ILMN_25544', 'ILMN_132331', 'ILMN_105017', 'ILMN_75398', 'ILMN_10519', 'ILMN_117209', 'ILMN_17234', 'ILMN_19244']

Expression probe dtype:
str

Annotation probe dtype:
str

Direct matches:
0

Annotation symbols:
[nan, 'LOC441782', 'JMJD1A', nan, nan, nan, 'NCOA3', nan, 'LOC389834', 'C17orf77']

Non-missing symbols:
35967


In [28]:
probe_test = "ILMN_1343291"

print(
    probe_annotation[
        probe_annotation["Search_Key"] == probe_test
    ][["Search_Key", "Symbol", "Entrez_Gene_ID"]]
)

Empty DataFrame
Columns: [Search_Key, Symbol, Entrez_Gene_ID]
Index: []


In [29]:
# Rebuild expression probe IDs cleanly
expression_reset = expression.reset_index()
expression_reset = expression_reset.rename(columns={"ID_REF": "Search_Key"})

expression_reset["Search_Key"] = (
    expression_reset["Search_Key"]
    .astype(str)
    .str.strip()
    .str.strip('"')
)

# Clean annotation probe IDs
probe_annotation["Search_Key"] = (
    probe_annotation["Search_Key"]
    .astype(str)
    .str.strip()
    .str.strip('"')
)

# Check exact examples
print("Expression probes:")
print(expression_reset["Search_Key"].head(10).tolist())

print("\nAnnotation probes:")
print(probe_annotation["Search_Key"].head(10).tolist())

# Check matching
matches = expression_reset["Search_Key"].isin(
    probe_annotation["Search_Key"]
)

print("\nExpression probes:", len(expression_reset))
print("Matching annotation probes:", matches.sum())
print("Unmatched probes:", (~matches).sum())
print("Match percentage:", round(matches.mean() * 100, 2), "%")

Expression probes:
['ILMN_1343291', 'ILMN_1343295', 'ILMN_1651199', 'ILMN_1651209', 'ILMN_1651210', 'ILMN_1651221', 'ILMN_1651228', 'ILMN_1651229', 'ILMN_1651230', 'ILMN_1651232']

Annotation probes:
['ILMN_89282', 'ILMN_35826', 'ILMN_25544', 'ILMN_132331', 'ILMN_105017', 'ILMN_75398', 'ILMN_10519', 'ILMN_117209', 'ILMN_17234', 'ILMN_19244']

Expression probes: 48803
Matching annotation probes: 0
Unmatched probes: 48803
Match percentage: 0.0 %


C:\Users\DELL\AppData\Local\Temp\ipykernel_2208\2440155005.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  expression_reset = expression.reset_index()


In [30]:
probe_test = expression_reset["Search_Key"].iloc[0]

print("Testing probe:", repr(probe_test))

print(
    probe_annotation[
        probe_annotation["Search_Key"] == probe_test
    ][["Search_Key", "Symbol", "Entrez_Gene_ID"]]
)

Testing probe: 'ILMN_1343291'
Empty DataFrame
Columns: [Search_Key, Symbol, Entrez_Gene_ID]
Index: []


In [31]:
# Compare the actual probe IDs character-by-character
expr_probe = expression_reset["Search_Key"].iloc[0]

print("Expression probe:", repr(expr_probe))
print("Length:", len(expr_probe))

print("\nFirst 10 annotation Search_Key values:")
for x in probe_annotation["Search_Key"].head(10):
    print(repr(x), "length =", len(x))

print("\nDoes expression probe exist in annotation?")
print(expr_probe in set(probe_annotation["Search_Key"]))

print("\nDoes it exist in ILMN_Gene?")
print(
    expr_probe in set(
        annotation["ILMN_Gene"]
        .dropna()
        .astype(str)
        .str.strip()
    )
)

Expression probe: 'ILMN_1343291'
Length: 12

First 10 annotation Search_Key values:
'ILMN_89282' length = 10
'ILMN_35826' length = 10
'ILMN_25544' length = 10
'ILMN_132331' length = 11
'ILMN_105017' length = 11
'ILMN_75398' length = 10
'ILMN_10519' length = 10
'ILMN_117209' length = 11
'ILMN_17234' length = 10
'ILMN_19244' length = 10

Does expression probe exist in annotation?
False

Does it exist in ILMN_Gene?
False


In [32]:
# Search the entire annotation for our probe ID
matches = annotation[
    annotation.astype(str).apply(
        lambda col: col.str.contains(
            "ILMN_1343291",
            regex=False,
            na=False
        )
    ).any(axis=1)
]

print("Rows containing ILMN_1343291:", len(matches))

display(matches)

Rows containing ILMN_1343291: 1


,Species,Source,Search_Key,Transcript,ILMN_Gene,Source_Reference_ID,RefSeq_ID,Unigene_ID,Entrez_Gene_ID,GI,...,Chromosome,Probe_Chr_Orientation,Probe_Coordinates,Cytoband,Definition,Ontology_Component,Ontology_Process,Ontology_Function,Synonyms,Obsolete_Probe_Id
4554,Homo sapiens,RefSeq,ILMN_137991,ILMN_137991,EEF1A1,NM_001402.4,NM_001402.4,NaN,1915.0,25453469,...,6,-,74284362-74284378:74284474-74284506,6q13c,Homo sapiens eukaryotic translation elongation...,cytoplasm [goid 5737] [evidence NAS]; cytoplas...,protein biosynthesis [goid 6412] [evidence IEA...,nucleotide binding [goid 166] [evidence IEA]; ...,PTI1; eEF1A-1; EEF1A; MGC16224; EF-Tu; EEF-1; ...,NaN


In [33]:
# Find exactly which annotation column contains our expression probe
probe_test = "ILMN_1343291"

for column in annotation.columns:
    hits = annotation[column].astype(str).str.strip().eq(probe_test).sum()
    if hits > 0:
        print(f"{column}: {hits} match(es)")

Probe_Id: 1 match(es)


In [34]:
# Show the complete annotation row for the matching probe
for column in annotation.columns:
    hits = annotation[column].astype(str).str.strip().eq(probe_test)
    
    if hits.any():
        print("\nMatching column:", column)
        display(annotation.loc[hits].T)


Matching column: Probe_Id


,4554
Species,Homo sapiens
Source,RefSeq
Search_Key,ILMN_137991
Transcript,ILMN_137991
ILMN_Gene,EEF1A1
Source_Reference_ID,NM_001402.4
RefSeq_ID,NM_001402.4
Unigene_ID,NaN
Entrez_Gene_ID,1915.0
GI,25453469


In [35]:
probe_test = "ILMN_1343291"

for column in annotation.columns:
    hits = annotation[column].astype(str).str.strip().eq(probe_test)
    
    if hits.any():
        row = annotation.loc[hits].iloc[0]
        
        print("MATCHING COLUMN:", column)
        print("Search_Key:", row["Search_Key"])
        print("ILMN_Gene:", row["ILMN_Gene"])
        print("Symbol:", row["Symbol"])
        print("Entrez_Gene_ID:", row["Entrez_Gene_ID"])

MATCHING COLUMN: Probe_Id
Search_Key: ILMN_137991
ILMN_Gene: EEF1A1
Symbol: EEF1A1
Entrez_Gene_ID: 1915.0


In [36]:
print("Expression probe:", repr(expression.index[0]))
print("Annotation ILMN_Gene examples:")
print(annotation["ILMN_Gene"].head(10).tolist())

Expression probe: 'ILMN_1343291'
Annotation ILMN_Gene examples:
['HS.388528', 'LOC441782', 'JMJD1A', 'HS.580150', 'HS.540210', 'HS.116531', 'NCOA3', 'HS.564563', 'LOC389834', 'C17ORF77']


In [37]:
# Correct probe-to-gene annotation
probe_annotation = annotation[
    ["Probe_Id", "Symbol", "Entrez_Gene_ID"]
].copy()

# Clean probe IDs
probe_annotation["Probe_Id"] = (
    probe_annotation["Probe_Id"]
    .astype(str)
    .str.strip()
    .str.strip('"')
)

# Clean gene symbols
probe_annotation["Symbol"] = (
    probe_annotation["Symbol"]
    .astype(str)
    .str.strip()
)

print("Annotation probes:", len(probe_annotation))
print("Probes with symbols:",
      (probe_annotation["Symbol"] != "nan").sum())

print("\nExample:")
display(probe_annotation.head())

Annotation probes: 48804
Probes with symbols: 48804

Example:


,Probe_Id,Symbol,Entrez_Gene_ID
0,ILMN_1825594,NaN,NaN
1,ILMN_1810803,LOC441782,441782.0
2,ILMN_1722532,JMJD1A,55818.0
3,ILMN_1884413,NaN,NaN
4,ILMN_1906034,NaN,NaN


In [38]:
# Rebuild probe annotation while preserving real missing values
probe_annotation = annotation[
    ["Probe_Id", "Symbol", "Entrez_Gene_ID"]
].copy()

probe_annotation["Probe_Id"] = (
    probe_annotation["Probe_Id"]
    .astype(str)
    .str.strip()
    .str.strip('"')
)

# Clean Symbol WITHOUT converting NaN into the string "nan"
probe_annotation["Symbol"] = probe_annotation["Symbol"].replace(
    ["", "nan", "NaN", "None", "---"],
    pd.NA
)

print("Total annotation probes:", len(probe_annotation))
print("Probes with valid symbols:",
      probe_annotation["Symbol"].notna().sum())
print("Probes without symbols:",
      probe_annotation["Symbol"].isna().sum())

Total annotation probes: 48804
Probes with valid symbols: 35967
Probes without symbols: 12837


In [39]:
expression_reset = expression.reset_index()

expression_reset = expression_reset.rename(
    columns={"ID_REF": "Probe_Id"}
)

expression_reset["Probe_Id"] = (
    expression_reset["Probe_Id"]
    .astype(str)
    .str.strip()
    .str.strip('"')
)

expression_annotated = expression_reset.merge(
    probe_annotation,
    on="Probe_Id",
    how="left"
)

print("Expression probes:", len(expression_reset))

print(
    "Successfully annotated:",
    expression_annotated["Symbol"].notna().sum()
)

print(
    "Without gene symbol:",
    expression_annotated["Symbol"].isna().sum()
)

print(
    "Annotation rate:",
    round(
        expression_annotated["Symbol"].notna().mean() * 100,
        2
    ),
    "%"
)

Expression probes: 48803
Successfully annotated: 35967
Without gene symbol: 12837
Annotation rate: 73.7 %


C:\Users\DELL\AppData\Local\Temp\ipykernel_2208\3376558612.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  expression_reset = expression.reset_index()


In [40]:
display(
    expression_annotated[
        ["Probe_Id", "Symbol", "Entrez_Gene_ID"]
    ].dropna(subset=["Symbol"]).head(10)
)

,Probe_Id,Symbol,Entrez_Gene_ID
0,ILMN_1343291,EEF1A1,1915.0
1,ILMN_1343295,GAPDH,2597.0
2,ILMN_1651199,LOC643334,643334.0
3,ILMN_1651209,SLC35E2,9906.0
4,ILMN_1651210,DUSP22,56940.0
5,ILMN_1651221,LOC642820,642820.0
6,ILMN_1651228,RPS28,6234.0
7,ILMN_1651229,IPO13,9670.0
8,ILMN_1651230,TESSP1,360226.0
9,ILMN_1651232,LOC653113,653113.0


In [41]:
# Keep only probes with valid gene symbols
mapped = expression_annotated.dropna(subset=["Symbol"]).copy()

# Expression columns
sample_columns = expression.columns.tolist()

# Calculate variance of each probe across all samples
mapped["probe_variance"] = mapped[sample_columns].var(axis=1)

print("Mapped probes:", len(mapped))
print("Unique genes:", mapped["Symbol"].nunique())

# How many genes have multiple probes?
probe_counts = mapped["Symbol"].value_counts()

print(
    "Genes represented by multiple probes:",
    (probe_counts > 1).sum()
)

print("\nMost highly represented genes:")
display(probe_counts.head(10))

Mapped probes: 35967
Unique genes: 25440
Genes represented by multiple probes: 6699

Most highly represented genes:


Symbol
DDX12        10
KIAA0692      9
LOC23117      8
DMD           8
AGL           8
PTGER3        8
PLEC1         8
CTNNB1        7
LOC202134     7
SDHAL2        7
Name: count, dtype: int64

In [42]:
# Rank probes within each gene by variance
best_probes = (
    mapped
    .sort_values(
        ["Symbol", "probe_variance"],
        ascending=[True, False]
    )
    .drop_duplicates(subset="Symbol", keep="first")
    .copy()
)

print("Genes after probe selection:", len(best_probes))

display(
    best_probes[
        ["Probe_Id", "Symbol", "probe_variance"]
    ].head(10)
)

Genes after probe selection: 25440


,Probe_Id,Symbol,probe_variance
27684,ILMN_1809034,15E1.2,0.921125
2103,ILMN_1660305,2'-PDE,0.206815
25018,ILMN_1792173,76P,0.314807
20343,ILMN_1762337,7A5,0.470894
41858,ILMN_2055271,A1BG,0.412481
47171,ILMN_2359168,A2BP1,0.151249
17718,ILMN_1745607,A2M,1.141820
43279,ILMN_2136495,A2ML1,0.222474
3765,ILMN_1668111,A3GALT2,0.066247
15988,ILMN_1735045,A4GALT,1.004717


In [43]:
gene_expression = (
    best_probes
    .set_index("Symbol")[sample_columns]
    .copy()
)

print("Final gene-expression matrix:", gene_expression.shape)

display(gene_expression.head())

Final gene-expression matrix: (25440, 166)


,GSM1941120,GSM1941121,GSM1941122,GSM1941123,GSM1941124,GSM1941125,GSM1941126,GSM1941127,GSM1941128,GSM1941129,...,GSM1941276,GSM1941277,GSM1941278,GSM1941279,GSM1941280,GSM1941281,GSM1941282,GSM1941283,GSM1941284,GSM1941285
Symbol,,,,,,,,,,,,,,,,,,,,,
15E1.2,5.382,6.375,5.773,5.469,4.807,3.700,6.691,6.750,4.621,5.170,...,5.548,5.274,5.909,6.912,5.365,5.436,4.485,5.647,4.070,5.752
2'-PDE,7.869,8.176,7.647,7.019,7.472,7.221,7.481,7.752,7.533,7.580,...,7.685,7.408,7.443,7.919,7.165,7.254,7.660,8.146,7.521,7.663
76P,7.238,6.438,7.859,6.934,6.382,7.473,7.464,7.127,7.071,7.391,...,7.116,6.976,7.739,7.881,7.336,7.615,7.353,7.840,8.369,6.998
7A5,3.632,5.752,3.916,3.678,3.154,3.365,3.472,3.838,3.838,3.472,...,3.644,3.667,3.498,3.945,3.485,4.650,3.621,3.733,3.536,3.597
A1BG,5.013,4.256,4.104,4.591,4.087,4.365,4.018,4.202,4.498,5.096,...,4.638,4.053,4.406,4.561,4.897,4.053,4.202,4.661,4.459,5.049


In [44]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

gene_expression.to_csv(
    processed_dir / "GSE75037_gene_expression.tsv.gz",
    sep="\t",
    compression="gzip"
)

metadata.to_csv(
    processed_dir / "GSE75037_metadata.tsv",
    sep="\t",
    index=False
)

print("Saved:")
print("✓ GSE75037_gene_expression.tsv.gz")
print("✓ GSE75037_metadata.tsv")

Saved:
✓ GSE75037_gene_expression.tsv.gz
✓ GSE75037_metadata.tsv
